# Run every model from one notebook

Each cell runs real scripts from this repo (`scripts/run_model.sh`, the Track B starters, `bench.plots`), so the files it
writes are the same as a batch run: `runs/`, `results/`, `figs/`, plus a full log per model in `logs/notebook/`.
While a model trains you see its last log lines and its **loss curves growing**; after each model finishes the
**leaderboard and comparison figure update** and its price-path plot appears.

**Where to run it**
- *On a GPU node (recommended):* PACE OnDemand → Jupyter, pick a GPU, kernel `gdl` (if it is not listed: `conda activate gdl; python -m ipykernel install --user --name gdl`). Use `MODE = "local"`.
- *From a login-node Jupyter or your laptop:* `MODE = "slurm"` submits one PACE job per model and the watch cell redraws every job's curves from its log file.

Run cells top to bottom. Everything is skippable: each section only needs its data to exist.

In [ ]:
# ---- settings -------------------------------------------------------------------------------------------
MODE   = "local"        # "local": run here (GPU Jupyter)   "slurm": submit PACE jobs and watch them
QUICK  = False          # True = tiny settings, a few minutes total, to check the plumbing end to end

DATA   = "bench_data/A"                     # synthetic Track A export (exact truth)
REAL   = "bench_data/real/megacap_tech"     # real export from scripts/real_data_setup.sh (login node, needs internet)
D2     = "bench_data/D2"                    # Track B option surfaces
MARKET = None                               # set below to the newest market snapshot for REAL, if there is one

import os, sys, glob
# notebooks/ -> repo root, so `python -m ...` and the imports below work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%load_ext autoreload
%autoreload 2
from bench import nb

snaps = sorted(glob.glob(f"bench_data/market/*_{os.path.basename(REAL)}"))
MARKET = snaps[-1] if snaps else None
print("repo:", os.getcwd(), "| market snapshot:", MARKET)
print("Track A models:", nb.TRACK_A); print("sets:", {k: v for k, v in nb.SETS.items() if k != 'all'})
print("Track B:", list(nb.TRACK_B))

## 1. Data
Creates the synthetic Track A data and the Track B surfaces if they are missing (a few minutes each). Real data needs internet, so make it on the login node with `bash scripts/real_data_setup.sh megacap_tech`.

In [ ]:
nb.ensure_data(DATA, D2, quick=QUICK)     # add n_hedge_paths=10000 for the final hedging numbers (re-export)
nb.show_data(DATA)
if os.path.exists(f"{REAL}/config.json"):
    nb.show_data(REAL)

## 2. Track A on synthetic data
Pick one model, a set (`baselines`, `thesis`, `generative`, `all`), or a list. Per-model extra arguments go in `extra`, e.g. `extra={'ar_diffusion': '--steps 40000'}`.

In [ ]:
if MODE == "local":
    dash = nb.run_trackA("baselines", DATA, quick=QUICK)        # seconds each; good first check

In [ ]:
if MODE == "local":
    nb.run_trackA("thesis", DATA, quick=QUICK, dashboard=dash)  # A1 AR+diffusion vs A2 tokens: the paper's main comparison

In [ ]:
if MODE == "local":
    nb.run_trackA("generative", DATA, quick=QUICK, dashboard=dash)   # RNADE, GAN, SDE-GAN, pure diffusion, flows

## 3. Track A on real data (backtest + market prices)
Same models trained on the real export; scored against what actually happened, and against today's option quotes if a market snapshot exists.

In [ ]:
if MODE == "local" and os.path.exists(f"{REAL}/config.json"):
    nb.run_trackA(["bsm", "garch", "ar_diffusion", "ar_token", "pure_diffusion", "path_flow"], REAL, market=MARKET, quick=QUICK)

## 4. Track B (option-implied models)
Baselines, EBM, flow, neural process, VAE, neural SDE, Schrödinger bridge. Table of held-out IV RMSE, B4 forward-start errors, and the Track B figures.

In [ ]:
if MODE == "local":
    nb.run_trackB("all", D2, quick=QUICK)       # or e.g. nb.run_trackB(["b1_ebm", "b4_schrodinger_bridge"], D2)

## 5. Report: all tables and figures
Runs `scripts/report.sh` (leaderboards, smiles, terminal densities, backtest, market) and shows every figure.

In [ ]:
nb.report(DATA)
if os.path.exists(f"results/{os.path.basename(REAL)}"):
    nb.report(REAL)

## 6. PACE batch mode (`MODE = "slurm"`)
Submits one GPU job per model, then redraws all jobs' loss curves and the leaderboard every 30 s until they finish. You can stop the cell any time; the jobs keep running, and re-running `watch` picks them back up.

In [ ]:
if MODE == "slurm":
    jobs = nb.submit_trackA("all", DATA)
    nb.watch(jobs, DATA)

In [ ]:
if MODE == "slurm" and os.path.exists(f"{REAL}/config.json"):
    jobs_real = nb.submit_trackA("all", REAL, market=MARKET)
    nb.watch(jobs_real, REAL)

## 7. Just look at results
No training: the leaderboard and figures from whatever has already run (here or in batch jobs).

In [ ]:
for d in (DATA, REAL):
    if os.path.exists(f"{d}/config.json"):
        nb.Dashboard(d).refresh()
display(nb.trackB_table()); display(nb.b4_table())